### SCM Surgery Datamart Exploration (`_exponent`.`_bronze_srgry_dmart`)

This notebook explores surgery datamart assets to assess readiness for:
- `drug_exposure`
- `condition_occurrence` / `condition_era`
- `episode` / `episode_event`

Run cells in order and copy results into `docs/scm_bronze_srgry_dmart_result_template.md`.


In [ ]:
%sql
USE CATALOG `_exponent`;
USE SCHEMA `_bronze_srgry_dmart`;


In [ ]:
%sql
-- Section A1: full table inventory
SHOW TABLES IN `_exponent`.`_bronze_srgry_dmart`;


In [ ]:
%sql
-- Section A3: candidate table names by domain hint
SELECT
  table_schema,
  table_name,
  CASE
    WHEN LOWER(table_name) RLIKE '(drug|med|medication|rx|pharm|order|mar|admin)' THEN 'drug_like'
    WHEN LOWER(table_name) RLIKE '(dx|diag|diagnosis|condition|problem|icd|snomed)' THEN 'condition_like'
    WHEN LOWER(table_name) RLIKE '(episode|encounter|visit|surg|surgery|procedure|case)' THEN 'episode_like'
    ELSE 'other'
  END AS domain_hint
FROM `_exponent`.`information_schema`.`tables`
WHERE table_schema = '_bronze_srgry_dmart'
  AND table_type IN ('BASE TABLE', 'VIEW')
  AND LOWER(table_name) RLIKE '(drug|med|medication|rx|pharm|order|mar|admin|dx|diag|diagnosis|condition|problem|icd|snomed|episode|encounter|visit|surg|surgery|procedure|case)'
ORDER BY domain_hint, table_name;


In [ ]:
%sql
-- Section A4: column inventory for candidate tables
SELECT
  table_name,
  column_name,
  data_type,
  ordinal_position
FROM `_exponent`.`information_schema`.`columns`
WHERE table_schema = '_bronze_srgry_dmart'
  AND LOWER(table_name) RLIKE '(drug|med|medication|rx|pharm|order|mar|admin|dx|diag|diagnosis|condition|problem|icd|snomed|episode|encounter|visit|surg|surgery|procedure|case)'
ORDER BY table_name, ordinal_position;


In [ ]:
%sql
-- Section B1-B4: key column discovery across all tables
SELECT table_name, column_name, data_type, 'person_key' AS key_type
FROM `_exponent`.`information_schema`.`columns`
WHERE table_schema = '_bronze_srgry_dmart'
  AND LOWER(column_name) RLIKE '(person|patient|client|mrn|empi|enterprise|subject)'
UNION ALL
SELECT table_name, column_name, data_type, 'visit_episode_key' AS key_type
FROM `_exponent`.`information_schema`.`columns`
WHERE table_schema = '_bronze_srgry_dmart'
  AND LOWER(column_name) RLIKE '(visit|encounter|episode|case|surg|admit|discharge)'
UNION ALL
SELECT table_name, column_name, data_type, 'date_time' AS key_type
FROM `_exponent`.`information_schema`.`columns`
WHERE table_schema = '_bronze_srgry_dmart'
  AND (LOWER(column_name) RLIKE '(date|dtm|datetime|time|start|end|onset|offset|admit|discharge|admin)'
       OR LOWER(data_type) RLIKE '(date|timestamp)')
UNION ALL
SELECT table_name, column_name, data_type, 'code_system' AS key_type
FROM `_exponent`.`information_schema`.`columns`
WHERE table_schema = '_bronze_srgry_dmart'
  AND LOWER(column_name) RLIKE '(code|concept|ndc|rxnorm|atc|gpi|icd|snomed|cpt|hcpcs|loinc|status|route|dose|unit)'
ORDER BY table_name, key_type, column_name;


### Section C: Drug probe pack

Replace placeholders with chosen candidate table/columns from A/B.
- `<drug_table>`
- `<person_col>`
- `<drug_code_col>`
- `<event_date_col>`
- `<status_col>`
- `<visit_col>`


In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT <person_col>) AS distinct_persons,
  SUM(CASE WHEN <person_col> IS NULL THEN 1 ELSE 0 END) AS null_person_rows,
  SUM(CASE WHEN <drug_code_col> IS NULL OR TRIM(CAST(<drug_code_col> AS STRING)) = '' THEN 1 ELSE 0 END) AS null_or_blank_drug_code_rows,
  SUM(CASE WHEN <event_date_col> IS NULL THEN 1 ELSE 0 END) AS null_event_date_rows,
  MIN(CAST(<event_date_col> AS DATE)) AS min_event_date,
  MAX(CAST(<event_date_col> AS DATE)) AS max_event_date
FROM `_exponent`.`_bronze_srgry_dmart`.<drug_table>;


### Section D/E: Condition and episode probes

Duplicate the prior probe cell and swap table/column placeholders for condition and episode candidates.
Use the same metric pattern: row volume, key completeness, code quality, date quality, and linkage IDs.


In [ ]:
%sql
-- Section F1: person overlap between drug and condition candidates
SELECT
  COUNT(DISTINCT d.person_key) AS drug_persons,
  COUNT(DISTINCT c.person_key) AS condition_persons,
  COUNT(DISTINCT CASE WHEN c.person_key IS NOT NULL THEN d.person_key END) AS overlap_persons
FROM (
  SELECT CAST(<drug_person_col> AS STRING) AS person_key
  FROM `_exponent`.`_bronze_srgry_dmart`.<drug_table>
  WHERE <drug_person_col> IS NOT NULL
) d
LEFT JOIN (
  SELECT DISTINCT CAST(<condition_person_col> AS STRING) AS person_key
  FROM `_exponent`.`_bronze_srgry_dmart`.<condition_table>
  WHERE <condition_person_col> IS NOT NULL
) c ON d.person_key = c.person_key;


In [ ]:
%sql
-- Section F2: visit/encounter overlap between drug and episode candidates
SELECT
  COUNT(DISTINCT d.visit_key) AS drug_visit_keys,
  COUNT(DISTINCT e.visit_key) AS episode_visit_keys,
  COUNT(DISTINCT CASE WHEN e.visit_key IS NOT NULL THEN d.visit_key END) AS overlap_visit_keys
FROM (
  SELECT CAST(<drug_visit_col> AS STRING) AS visit_key
  FROM `_exponent`.`_bronze_srgry_dmart`.<drug_table>
  WHERE <drug_visit_col> IS NOT NULL
) d
LEFT JOIN (
  SELECT DISTINCT CAST(<episode_visit_col> AS STRING) AS visit_key
  FROM `_exponent`.`_bronze_srgry_dmart`.<episode_table>
  WHERE <episode_visit_col> IS NOT NULL
) e ON d.visit_key = e.visit_key;
